In [1]:
import csv
import hashlib
from collections import defaultdict
from datetime import date
from pathlib import Path

PROJECT = Path(r"C:\Users\LENOVO\OneDrive\Desktop\科研-王霞\文本数据专题")
DATA = PROJECT / "数据" / "文本原始数据"
PAPERS = ("经济日报", "经济参考报", "21世纪经济报道")
ENCODING = "utf-8-sig"

csv.field_size_limit(32 * 1024 * 1024)


def article_key(title, body):
    """比较文章前，把连续空格、换行和制表符统一成一个空格。"""
    title = " ".join(title.split())
    body = " ".join(body.split())
    if not title or not body:
        return None
    content = f"{title}\0{body}".encode("utf-8")
    return hashlib.sha256(content).digest()


def scan_file(path, supplement_keys, is_supplement):
    days = set()
    rows = bad_width = bad_date = matched = 0

    print(f"正在检查 {path.name}", flush=True)
    with path.open(encoding=ENCODING, newline="") as file:
        reader = csv.reader(file, strict=True)
        headers = next(reader, [])
        required = {"date", "title", "text"}
        missing = required.difference(headers)
        if missing:
            raise ValueError(f"{path.name} 缺少字段 {sorted(missing)}")

        date_col = headers.index("date")
        title_col = headers.index("title")
        body_col = headers.index("text")

        for row in reader:
            if not row:
                continue
            rows += 1

            if len(row) != len(headers):
                bad_width += 1
                continue

            try:
                day = date.fromisoformat(row[date_col].strip())
            except ValueError:
                bad_date += 1
                continue
            days.add(day)

            if is_supplement:
                key = article_key(row[title_col], row[body_col])
                if key is not None:
                    supplement_keys[day].add(key)
            else:
                candidates = supplement_keys.get(day)
                if candidates:
                    key = article_key(row[title_col], row[body_col])
                    if key in candidates:
                        candidates.remove(key)
                        matched += 1

    summary = {
        "文件": path.name,
        "编码": ENCODING,
        "表头": " | ".join(headers),
        "最早日期": min(days, default=""),
        "最晚日期": max(days, default=""),
        "记录数": rows,
        "列数异常条数": bad_width,
        "日期无法解析条数": bad_date,
    }
    print(f"  {rows:,} 条，日期 {summary['最早日期']} 至 {summary['最晚日期']}")
    return summary, days, matched


def save_csv(path, rows):
    with path.open("w", encoding=ENCODING, newline="") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    print(f"已保存 {path}")


file_rows = []
overlap_rows = []

for paper in PAPERS:
    supplement_keys = defaultdict(set)

    # 先读补充文件，仅保留按日期索引的文章指纹。
    supplement, supplement_days, _ = scan_file(
        DATA / f"{paper}-21年后补充.csv", supplement_keys, is_supplement=True
    )
    original, original_days, matched = scan_file(
        DATA / f"{paper}.csv", supplement_keys, is_supplement=False
    )
    file_rows.extend([original, supplement])

    overlap_start = overlap_end = ""
    status = "至少一个文件没有有效日期"
    if original_days and supplement_days:
        start = max(min(original_days), min(supplement_days))
        end = min(max(original_days), max(supplement_days))
        status = "日期范围无交集"
        if start <= end:
            overlap_start, overlap_end = start, end
            status = "日期范围有交集"

    overlap_rows.append({
        "报纸": paper,
        "日期范围交集起点": overlap_start,
        "日期范围交集终点": overlap_end,
        "实际共同刊发天数": len(original_days & supplement_days),
        "完全重复篇数": matched,
        "日期范围状态": status,
    })

save_csv(PROJECT / "新闻数据情况表.csv", file_rows)
save_csv(PROJECT / "新闻补充数据重叠表.csv", overlap_rows)

正在检查 经济日报-21年后补充.csv
  67,654 条，日期 2021-01-01 至 2024-12-31
正在检查 经济日报.csv
  442,290 条，日期 2000-01-01 至 2022-04-08
正在检查 经济参考报-21年后补充.csv
  34,080 条，日期 2021-01-04 至 2024-12-31
正在检查 经济参考报.csv
  313,618 条，日期 2000-01-10 至 2022-11-04
正在检查 21世纪经济报道-21年后补充.csv
  31,759 条，日期 2021-01-01 至 2024-12-31
正在检查 21世纪经济报道.csv
  258,039 条，日期 2001-01-01 至 2021-11-18
已保存 C:\Users\LENOVO\OneDrive\Desktop\科研-王霞\文本数据专题\新闻数据情况表.csv
已保存 C:\Users\LENOVO\OneDrive\Desktop\科研-王霞\文本数据专题\新闻补充数据重叠表.csv


捕获重复文本

In [ ]:
import pandas as pd

keys = ["date", "title", "text"]
df1 = pd.read_csv(PROJECT / "21世纪经济报道.csv", encoding=ENCODING)
df11 = pd.read_csv(PROJECT / "21世纪经济报道-21年后补充.csv", encoding=ENCODING)
df111 = df1[keys].merge(df11[keys], how="inner", on=keys).drop_duplicates()

df111.to_csv(PROJECT / "21世纪经济报道-21年后补充-重复.csv", index=False, encoding=ENCODING)